In [34]:
import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib version:", matplotlib.__version__)

Matplotlib version: 3.9.2


In [35]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
df = pd.read_csv("student_performance.csv")

# 1. STRUCTURE
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape: (1200, 9)
Missing values:
 student_id         0
county             0
school_type        0
gender             0
attendance_rate    0
study_hours        0
math               0
english            0
science            0
dtype: int64
Duplicate rows: 0


In [36]:
# 2. UNIVARIATE
num_cols = ["attendance_rate", "study_hours", "math", "english", "science"]
uni_num = df[num_cols].describe().T
uni_num["median"] = df[num_cols].median()
uni_num["iqr"] = uni_num["75%"] - uni_num["25%"]
uni_num["skew"] = df[num_cols].skew()
display(uni_num[["mean", "median", "std", "iqr", "min", "max", "skew"]].round(2))

# Outliers in study_hours (IQR method)
q1, q3 = df["study_hours"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
outliers = df[df["study_hours"] > upper_fence]
print(f"study_hours Upper Fence: {upper_fence:.2f} | Outliers: {len(outliers)}")

for col in ["county", "school_type", "gender"]:
    display(pd.DataFrame({
        "count": df[col].value_counts(),
        "share_%": (df[col].value_counts(normalize=True) * 100).round(1)
    }))

,mean,median,std,iqr,min,max,skew
attendance_rate,0.80,0.82,0.12,0.17,0.33,1.0,-0.86
study_hours,5.08,4.50,3.03,3.80,0.20,23.3,1.17
math,71.89,72.00,10.86,14.00,37.00,100.0,0.10
english,74.52,74.00,10.49,14.00,47.00,100.0,0.16
science,69.61,69.00,12.02,17.00,35.00,100.0,0.11


study_hours Upper Fence: 12.40 | Outliers: 29


,count,share_%
county,,
Mombasa,212,17.7
Kisumu,208,17.3
Nyeri,203,16.9
Nakuru,202,16.8
Turkana,191,15.9
Nairobi,184,15.3


,count,share_%
school_type,,
Public,866,72.2
Private,334,27.8


,count,share_%
gender,,
F,612,51.0
M,588,49.0


In [37]:
# 3. DERIVED COLUMNS: average_score & performance_band
df["average_score"] = df[["math", "english", "science"]].mean(axis=1)
df["performance_band"] = pd.cut(
    df["average_score"],
    bins=[-np.inf, 60, 75, np.inf],
    labels=["At Risk", "Meeting", "Exceeding"],
    right=False,
)

In [38]:
# 4. BIVARIATE: Correlations & Subgroups
corr = df[["attendance_rate", "study_hours", "math", "english", "science", "average_score"]].corr()
display(corr.round(3))

for grp in ["gender", "school_type", "county"]:
    sub = df.groupby(grp, as_index=False).agg(
        students=("student_id", "count"),
        mean_score=("average_score", "mean"),
        median_score=("average_score", "median"),
        at_risk_count=("performance_band", lambda s: (s == "At Risk").sum()),
        at_risk_pct=("performance_band", lambda s: (s == "At Risk").mean() * 100),
    ).sort_values("at_risk_pct", ascending=False).round(2)
    display(sub)

,attendance_rate,study_hours,math,english,science,average_score
attendance_rate,1.000,-0.029,0.300,0.327,0.260,0.359
study_hours,-0.029,1.000,0.560,0.624,0.550,0.704
math,0.300,0.560,1.000,0.559,0.459,0.814
english,0.327,0.624,0.559,1.000,0.502,0.827
science,0.260,0.550,0.459,0.502,1.000,0.815
average_score,0.359,0.704,0.814,0.827,0.815,1.000


,gender,students,mean_score,median_score,at_risk_count,at_risk_pct
0,F,612,72.10,71.67,57,9.31
1,M,588,71.91,71.33,42,7.14


,school_type,students,mean_score,median_score,at_risk_count,at_risk_pct
1,Public,866,70.46,69.67,94,10.85
0,Private,334,76.01,75.33,5,1.50


,county,students,mean_score,median_score,at_risk_count,at_risk_pct
2,Nairobi,184,71.96,72.17,17,9.24
1,Mombasa,212,72.17,72.17,19,8.96
4,Nyeri,203,71.94,71.33,18,8.87
0,Kisumu,208,73.07,72.50,17,8.17
5,Turkana,191,70.97,70.00,15,7.85
3,Nakuru,202,71.84,70.83,13,6.44


In [39]:


#  5 Compare average scores and At Risk share across gender, school_type, and county
for group_col in ["gender", "school_type", "county"]:
    summary = (
        df.groupby(group_col, as_index=False)
        .agg(
            students=("student_id", "count"),
            mean_score=("average_score", "mean"),
            median_score=("average_score", "median"),
            std_score=("average_score", "std"),
            at_risk_count=("performance_band", lambda s: (s == "At Risk").sum()),
            at_risk_pct=("performance_band", lambda s: (s == "At Risk").mean() * 100),
        )
        .sort_values("at_risk_pct", ascending=False)
        .round(2)
    )
    print(f"\n=== Average Scores by {group_col} ===")
    print(summary.to_string(index=False))


=== Average Scores by gender ===
gender  students  mean_score  median_score  std_score  at_risk_count  at_risk_pct
     F       612       72.10         71.67       9.17             57         9.31
     M       588       71.91         71.33       9.04             42         7.14

=== Average Scores by school_type ===
school_type  students  mean_score  median_score  std_score  at_risk_count  at_risk_pct
     Public       866       70.46         69.67       9.00             94        10.85
    Private       334       76.01         75.33       8.12              5         1.50

=== Average Scores by county ===
 county  students  mean_score  median_score  std_score  at_risk_count  at_risk_pct
Nairobi       184       71.96         72.17       9.35             17         9.24
Mombasa       212       72.17         72.17       8.35             19         8.96
  Nyeri       203       71.94         71.33       9.13             18         8.87
 Kisumu       208       73.07         72.50       9.54

In [40]:
# 6 . MULTIVARIATE: Pivot Tables Testing School-Type Gap# 
df["attendance_band"] = pd.cut(
    df["attendance_rate"], bins=[0, 0.70, 0.80, 0.90, 1.00], labels=["<=0.70", "0.71-0.80", "0.81-0.90", ">0.90"]
)
df["study_quartile"] = pd.qcut(df["study_hours"], q=4, labels=["Q1 (Low)", "Q2", "Q3", "Q4 (High)"])

piv_att = pd.pivot_table(df, values="average_score", index="attendance_band", columns="school_type", aggfunc="mean", observed=False)
piv_att["Gap (Private - Public)"] = piv_att["Private"] - piv_att["Public"]
display(piv_att.round(2))

piv_study = pd.pivot_table(df, values="average_score", index="study_quartile", columns="school_type", aggfunc="mean", observed=False)
piv_study["Gap (Private - Public)"] = piv_study["Private"] - piv_study["Public"]
display(piv_study.round(2))

school_type,Private,Public,Gap (Private - Public)
attendance_band,,,
<=0.70,71.51,64.94,6.58
0.71-0.80,75.50,68.82,6.68
0.81-0.90,77.04,72.43,4.61
>0.90,79.39,74.19,5.21


school_type,Private,Public,Gap (Private - Public)
study_quartile,,,
Q1 (Low),69.51,63.82,5.69
Q2,74.92,67.12,7.80
Q3,76.18,71.18,5.00
Q4 (High),83.65,80.06,3.59


In [41]:
band_categories = ["At Risk", "Meeting", "Exceeding"]

df["performance_band"] = pd.Categorical(
    pd.cut(
        df["average_score"],
        bins=[-np.inf, 60, 75, np.inf],
        labels=band_categories,
        right=False,
    ),
    categories=band_categories,
    ordered=True,
)

# Retained across value_counts, crosstabs, and groupby calls:
band_counts = pd.crosstab(df["school_type"], df["performance_band"], dropna=False).reindex(
    columns=band_categories, fill_value=0
)

One-Sentence Limitation for the study_hours Trend LineTrend-Line Interpretation: Fitting np.polyfit(study_hours, average_score, 1) yields a slope of 2.12 percentage points per weekly study hour (intercept = 61.26, $R^2 = 0.4954$).One-Sentence Limitation: Because study_hours explains less than half of the variance in student scores ($R^2 = 0.4954$) and ignores attendance, school resources, and score ceilings at $100\%$, using this single-variable linear trend to predict an individual student's exam score yields wide prediction errors.

In [44]:
# 7. HYPOTHESIS / TRENDS: np. polyfit
import numpy as np
import pandas as pd

# Load data and compute average_score
df = pd.read_csv("student_performance.csv")
df["average_score"] = df[["math", "english", "science"]].mean(axis=1)

# Fit linear trend line of average_score against study_hours
x = df["study_hours"].to_numpy()
y = df["average_score"].to_numpy()

slope, intercept = np.polyfit(x, y, deg=1)
y_pred = slope * x + intercept

# Calculate R-squared
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print(f"Slope:     {slope:.4f}")
print(f"Intercept: {intercept:.4f}")
print(f"R²:        {r_squared:.4f}")

Slope:     2.1166
Intercept: 61.2595
R²:        0.4954


In [45]:
#  8 Create the 'performance_band' column based on average_score
# Students with average_score < 60 are classified as "At Risk"
df["performance_band"] = df["average_score"].apply(lambda x: "At Risk" if x < 60 else "Not At Risk")

# 8 THREE PANDAS .plot() CHARTS SAVED TO figures/
ax1 = df["average_score"].plot(kind="hist", bins=20, edgecolor="white", color="#2B6CB0", title="Distribution of Student Average Scores (n = 1,200)")
ax1.set_xlabel("Average Score (%)")
ax1.axvline(60, color="#C53030", linestyle="--", linewidth=1.5, label="At Risk Cutoff (< 60)")
ax1.legend()
plt.tight_layout()
plt.savefig("figures/eda_score_hist.png", dpi=150)
plt.close()

at_risk_county = (df["performance_band"] == "At Risk").groupby(df["county"]).mean().mul(100).sort_values()
ax2 = at_risk_county.plot(kind="barh", color="#2B6CB0", title="Share of At-Risk Learners (< 60%) by County")
ax2.set_xlabel("At-Risk Learners (%)")
plt.tight_layout()
plt.savefig("figures/eda_county_at_risk_bar.png", dpi=150)
plt.close()

ax3 = df.boxplot(column="average_score", by="school_type", grid=False)
ax3.set_title("Average Score Distribution by School Type")
plt.suptitle("")
ax3.set_ylabel("Average Score (%)")
ax3.set_xlabel("School Type")
plt.tight_layout()
plt.savefig("figures/eda_school_type_box.png", dpi=150)
plt.close()

Findings

1. **Overall At-Risk Population:** Across **1,200** learners, **8.2% (99 students)** fall into the **At Risk** band (`< 60%`), while **56.0% (672)** are **Meeting** (`60–74%`) and **35.8% (429)** are **Exceeding** (`≥ 75%`).
2. **Primary Behavioral Drivers:** Weekly `study_hours` has the strongest correlation with `average_score` ($r = 0.704$), followed by `attendance_rate` ($r = 0.359$), and the two habits operate independently ($r = -0.029$).
3. **Public School Concentration:** Public schools account for **94.9% (94 of 99)** of all At-Risk learners, recording an At-Risk rate of **10.85%** compared to **1.50%** (`5 of 334`) in Private schools.
4. **Persistent Structural School-Type Gap:** Private learners outscore Public learners by **+5.55 percentage points** overall (**76.01%** vs. **70.46%**), and a **+3.59 to +7.80 point gap persists** across every attendance band and study-hour quartile.
5. **County Disparities & Study Slope:** **Nairobi** has the highest share of At-Risk learners (**9.24%**, `17/184`) while **Turkana** posts the lowest county mean score (**70.97%**); across all students, each extra weekly study hour is associated with **+2.12 percentage points** ($R^2 = 0.4954$).

Recommendations

1. **Target Public Schools in High-Risk & Low-Mean Counties First:** Direct the first phase of remedial tutoring and instructional material grants exclusively to **Public schools**—prioritizing campuses in **Nairobi (9.24% At Risk)**, **Mombasa (8.96% At Risk)**, and **Turkana (lowest mean score at 70.97%)** where 95% of At-Risk learners are enrolled.
2. **Pair Attendance Early-Warning Alerts with Supervised Study Blocks:** Automatically enroll any learner whose `attendance_rate` drops below **0.80** or whose weekly `study_hours` fall in the bottom quartile (**< 2.9 hours/week**, where Public learners average **63.82%**) into structured after-school study halls with guardian follow-ups.

Limitations

This analysis examines a single cross-sectional snapshot of 1,200 learners, so observed relationships (including the +2.12 points per study hour slope) capture statistical associations rather than proven causal impacts. Critical unmeasured variables—such as household socioeconomic status, textbook and digital device access, teacher-to-student ratios, and prior baseline exam scores—are absent from the dataset and likely explain both part of the persistent Private–Public performance gap and the remaining 50.5% of unexplained score variance ($1 - R^2$).